# 22 — Extract user_states for ALL train sessions (W4 prerequisite)

Per RecSys_Challenge_Plan §6.3 (W4 envelope augmentation P1 #17). One-time job: run StateTracker on all ~15k train sessions × ~7 turns ≈ 100k turns and save to Drive at `experiments_cache/state/{session_id}__{turn}.json`.

**Why this is needed:** `scripts/augment_envelope.py` wraps every train turn's gold response in `<user_state>{state}</user_state><response>{response}</response>`. The state must be extracted from `(user_query, history)` only — never peeking at the gold response (zero-leakage). This script does it once for the entire train split so the augmenter runs locally in seconds.

**Wall time on A100:** ~3 hours (100k turns × ~100 ms/turn batched).

**Output:** ~100k JSON files at `MyDrive/recsys2026-cache/experiments_cache/state/`. Total ~50 MB. Persists across runtime resets thanks to the Drive symlink.

**Run frequency:** ONCE. After this completes, every future W4/W5/W6 envelope-augmentation run reads from the cache.

In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!git log -1 --pretty=format:'commit:  %h%nsubject: %s'

In [ ]:
# 2b) Mount Drive + persistent caches.
import os, shutil
from google.colab import drive

try: drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}\nretrying ...')
    try: drive.flush_and_unmount()
    except Exception: pass
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

print(f'state cache will land at: {EXPECTED_CACHE}/state/')

In [ ]:
# 3) Install minimal deps. State extraction needs torch + transformers + datasets.
!pip install -q --upgrade transformers datasets pandas tqdm
!python -c 'import torch, transformers; print("torch", torch.__version__, "cuda", torch.cuda.is_available())'

In [ ]:
# 4) Pre-flight: how many states exist already? (for resumability)
from pathlib import Path
state_dir = Path(EXPECTED_CACHE) / 'state'
state_dir.mkdir(parents=True, exist_ok=True)
existing = list(state_dir.glob('*.json'))
print(f'already cached: {len(existing):,} state files')
print(f'expected target: ~100,000 (15k sessions × ~7 turns/session with GPA labels)')

In [ ]:
# 5) Run state extraction on the entire train split.
# This is a thin wrapper around scripts/smoke_state_tracker.py — but with
# n_sessions=15000 (the full split) and producing files keyed by
# (session_id, turn). The StateTracker's own caching handles resumability —
# if a session+turn was already processed, it loads from Drive instead of
# re-extracting.

import sys, time
import pandas as pd
import torch
from datasets import load_dataset

sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
from mcrs.lm_modules.llama import LLAMA_MODEL
from mcrs.query_rewriters.state_tracker import StateTracker

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
PROMPT_PATH = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts/state_extraction.txt'
CACHE_DIR = EXPECTED_CACHE  # '/content/recsys2026/music-crs-baselines/experiments/cache'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if device == 'cuda' else torch.float32

print(f'loading {MODEL} on {device}...')
lm = LLAMA_MODEL(model_name=MODEL, device=device, attn_implementation='sdpa', dtype=dtype)
tracker = StateTracker(
    lm=lm, prompt_path=PROMPT_PATH, cache_dir=CACHE_DIR,
    max_new_tokens=96,
)
print(f'StateTracker ready (cache → {tracker.cache_root})')

In [ ]:
# 6) Walk train turns and extract.
# Total: ~15k sessions × ~7 turns = ~100k extractions.
# Resumability: tracker._load_cached() hits the Drive cache on repeat sessions.

from tqdm import tqdm

tr = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
print(f'train sessions: {len(tr):,}')

MAX_HISTORY_TURNS = 4

def history_text_for(convos, turn_n):
    df = pd.DataFrame(convos)
    prior = df[df['turn_number'] < turn_n]
    if prior.empty: return ''
    lines = []
    for _, t in prior.tail(MAX_HISTORY_TURNS * 3).iterrows():
        role = 'assistant' if t['role'] == 'music' else t['role']
        content = str(t['content'])[:200]
        lines.append(f'{role}: {content}')
    return '\n'.join(lines)

t_start = time.time()
n_extracted = 0
n_cache_hit_initial = tracker.stats['cache_hits']
n_cache_hit_running = 0

for sess in tqdm(tr.to_list(), desc='train sessions'):
    sid = sess['session_id']
    convos = sess.get('conversations') or []
    if not convos: continue
    df = pd.DataFrame(convos)
    # Skip turns that don't have a GPA label (we only train on labeled turns).
    gpa_map = {int(x['turn_number']): x.get('goal_progress_assessment')
               for x in (sess.get('goal_progress_assessments') or [])}
    for turn_n in sorted(df['turn_number'].unique()):
        tn = int(turn_n)
        if gpa_map.get(tn) is None:
            continue  # No GPA label = skip
        tdf = df[df['turn_number'] == tn]
        user_row = tdf[tdf['role'] == 'user']
        if user_row.empty: continue
        user_query = str(user_row.iloc[0]['content'])
        history = history_text_for(convos, tn)
        # extract() handles caching internally — calls model only on cache miss
        tracker.extract(sid, tn, user_query, history)
        n_extracted += 1

elapsed = time.time() - t_start
rep = tracker.report()
print(f'\nExtraction complete in {elapsed/60:.1f} min')
print(f'  total calls:           {rep["calls"]:,}')
print(f'  cache hits:            {rep["cache_hits"]:,}')
print(f'  ok first try:          {rep["ok_first_try"]:,}')
print(f'  ok after retry:        {rep["ok_after_retry"]:,}')
print(f'  fallback to prior:     {rep["fallback_to_prior"]:,}')
print(f'  drop (no fallback):    {rep["drop"]:,}')
print(f'  parse_validity (excl cache): {rep["parse_validity_excl_cache"]:.4f}')

In [ ]:
# 7) Verify final cache size + sanity sample.
from pathlib import Path
import json
state_dir = Path(EXPECTED_CACHE) / 'state'
files = list(state_dir.glob('*.json'))
print(f'cached states: {len(files):,}')
print(f'expected:      ~100,000')

# Sample 3 random states.
import random
for fp in random.sample(files, min(3, len(files))):
    with fp.open() as f: state = json.load(f)
    print(f'\n{fp.name}')
    for k, v in state.items():
        print(f'  {k}: {v}')

## After this run

- The state cache is now populated for the full train split.
- **Locally**, you can now run:
  ```bash
  python scripts/build_reward_dataset.py            # ~60-80k labeled turns → reward_train.parquet
  python scripts/augment_envelope.py                # Wraps text_b in <user_state>...<response>
  python scripts/build_trl_datasets.py --in data/reward_train_envelope.parquet  # → kto.parquet
  ```
- Then run `colab/30_train_responder_kto.ipynb` for the actual W4 training.